In [1]:
!pip install lm-eval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.1 MB/s eta 0:00:00


In [2]:
# Download the pre-trained model of choice
import gdown

# https://drive.google.com/file/d/14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3/view?usp=sharing # V3 best
GOOGLE_DRIVE_FILE_ID = '14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3'
OUTPUT_FILE_NAME = 'model.pth'

gdown.download(id=GOOGLE_DRIVE_FILE_ID, output=OUTPUT_FILE_NAME, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3
From (redirected): https://drive.google.com/uc?id=14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3&confirm=t&uuid=8c34be45-667c-4a71-bf87-95de00226a03
To: /content/model.pth
100%|██████████| 349M/349M [00:07<00:00, 43.8MB/s]


'model.pth'

In [3]:
# Load model and tokenizer

import torch
import torch.nn as nn
import math
from transformers import GPT2Tokenizer


class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        N, seq_len, _ = x.shape

        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)

        values = values.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)
        keys = keys.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)
        queries = queries.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)

        energy = torch.matmul(queries, keys.transpose(-2, -1))
        energy = energy / math.sqrt(self.head_dim)

        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        energy = energy.masked_fill(mask == 0, float("-inf"))

        attention = torch.softmax(energy, dim=-1)
        out = torch.matmul(attention, values)

        out = out.transpose(1, 2).contiguous().reshape(N, seq_len, self.embed_size)
        return self.fc_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Linear(4 * embed_size, embed_size)
        )

    def forward(self, x):
        x = self.norm1(x + self.attention(x))
        x = self.norm2(x + self.ff(x))
        return x

class GPT1(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, heads, max_len):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_len, embed_size)

        self.layers = nn.ModuleList(
            [TransformerBlock(embed_size, heads) for _ in range(num_layers)]
        )

        self.norm = nn.LayerNorm(embed_size)
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        N, seq_len = x.shape
        positions = torch.arange(0, seq_len).expand(N, seq_len).to(x.device)

        x = self.token_embedding(x) + self.position_embedding(positions)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        return self.fc_out(x)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT1(
    vocab_size = tokenizer.vocab_size,
    embed_size = 256,
    num_layers = 4,
    heads = 4,
    max_len = 512
).to(device)

In [5]:
# Load the state model from the downloaded checkpoint

state_dict = torch.load(OUTPUT_FILE_NAME, map_location=device)
model.load_state_dict(state_dict['model_state_dict'])
model.eval() # Uncommented model.eval()

GPT1(
  (token_embedding): Embedding(50257, 256)
  (position_embedding): Embedding(512, 256)
  (layers): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attention): SelfAttention(
        (values): Linear(in_features=256, out_features=256, bias=True)
        (keys): Linear(in_features=256, out_features=256, bias=True)
        (queries): Linear(in_features=256, out_features=256, bias=True)
        (fc_out): Linear(in_features=256, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (fc_out): Linear(in_features=256, out_features=50257, bias=True)
)

In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [7]:
# MMLU Class Wrapper Setup

import torch
import torch.nn.functional as F
from lm_eval.api.model import LM
from lm_eval import simple_evaluate

class GPT1Wrapper(LM):
    def __init__(self, model, tokenizer, max_len, device="cuda", batch_size=64):
        super().__init__()
        self._device = device
        self.model = model.to(device)
        self.model.eval()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.batch_size = batch_size
        self.pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def loglikelihood(self, requests):
        encoded = []
        for i, req in enumerate(requests):
            context, continuation = req.args
            context_enc = self.tokenizer.encode(context) if context else [self.tokenizer.eos_token_id]
            continuation_enc = self.tokenizer.encode(continuation)
            full_enc = (context_enc + continuation_enc)[-self.max_len:]
            cont_len = min(len(continuation_enc), len(full_enc))
            encoded.append((i, full_enc, cont_len))

        encoded.sort(key=lambda x: len(x[1]))  # sort by length -> minimal padding per batch
        results = [None] * len(requests)

        for start in range(0, len(encoded), self.batch_size):
            batch = encoded[start:start + self.batch_size]
            max_l = max(len(x[1]) for x in batch)

            input_ids = torch.full((len(batch), max_l), self.pad_id, dtype=torch.long)
            attn_mask = torch.zeros((len(batch), max_l), dtype=torch.long)
            for b, (_, seq, _) in enumerate(batch):
                input_ids[b, :len(seq)] = torch.tensor(seq)
                attn_mask[b, :len(seq)] = 1

            input_ids, attn_mask = input_ids.to(self._device), attn_mask.to(self._device)

            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
                # Removed attention_mask from the model call as GPT1 does not accept it
                logits = self.model(input_ids)

            logits = logits[:, :-1, :].float()
            targets = input_ids[:, 1:]
            log_probs = F.log_softmax(logits, dim=-1)
            tok_lp = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)

            for b, (orig_i, seq, cont_len) in enumerate(batch):
                L = len(seq)
                cont_lp = tok_lp[b, L - 1 - cont_len: L - 1]
                greedy = logits[b, L - 1 - cont_len: L - 1].argmax(dim=-1)
                actual = targets[b, L - 1 - cont_len: L - 1]
                results[orig_i] = (cont_lp.sum().item(), bool(torch.equal(greedy, actual)))

        return results

    def loglikelihood_rolling(self, requests):
        raise NotImplementedError

    def generate_until(self, requests):
        raise NotImplementedError

In [8]:
MAX_LEN = 512

lm_wrapper = GPT1Wrapper(model, tokenizer, max_len=MAX_LEN) # Use the global 'device' variable


In [9]:
from lm_eval import simple_evaluate
from lm_eval.tasks import TaskManager

# Confirm what "mmlu" expands to
tm = TaskManager()
mmlu_subtasks = tm.match_tasks(["mmlu_*"])
print(f"Number of MMLU subject tasks: {len(mmlu_subtasks)}")

Number of MMLU subject tasks: 2072


In [10]:
results = simple_evaluate(
    model=lm_wrapper,
    tasks=["mmlu"],
    num_fewshot=0,
    limit=100,       # 100 questions PER subject (57 subjects total)
    batch_size=64,
)

print(results["results"])

# Sanity-check the per-subject sample counts actually used
for task_name, res in results["results"].items():
    if task_name != "mmlu":
        n = results["n-samples"].get(task_name, {}).get("effective", None)
        print(task_name, "->", n, "questions")

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

abstract_algebra/validation-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 3.73kB            

...

Generating test split:   0%|          | 0/171 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/19 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

100%|██████████| 171/171 [00:00<00:00, 630.37it/s]


{'mmlu_abstract_algebra': {'name': 'mmlu_abstract_algebra', 'alias': 'abstract_algebra', 'sample_len': 100, 'acc,none': 0.22, 'acc_stderr,none': 0.041633319989322654}, 'mmlu_anatomy': {'name': 'mmlu_anatomy', 'alias': 'anatomy', 'sample_len': 135, 'acc,none': 0.18518518518518517, 'acc_stderr,none': 0.03355677216313144}, 'mmlu_astronomy': {'name': 'mmlu_astronomy', 'alias': 'astronomy', 'sample_len': 152, 'acc,none': 0.17763157894736842, 'acc_stderr,none': 0.031103182383123377}, 'mmlu_college_biology': {'name': 'mmlu_college_biology', 'alias': 'college_biology', 'sample_len': 144, 'acc,none': 0.2569444444444444, 'acc_stderr,none': 0.03653946969442102}, 'mmlu_college_chemistry': {'name': 'mmlu_college_chemistry', 'alias': 'college_chemistry', 'sample_len': 100, 'acc,none': 0.2, 'acc_stderr,none': 0.04020151261036849}, 'mmlu_college_computer_science': {'name': 'mmlu_college_computer_science', 'alias': 'college_computer_science', 'sample_len': 100, 'acc,none': 0.25, 'acc_stderr,none': 0.04